In [1]:
import os
os.environ["KERAS_BACKEND"] = "torch"

In [3]:
import pandas as pd
import numpy as np
import pyarrow
from src_RN import *
import itertools, gc

from sklearn.model_selection import  train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, MinMaxScaler
from sklearn.model_selection import StratifiedKFold

import keras
from keras.models import Sequential
from keras.layers import Dense, Input, Dropout
from keras import optimizers

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras import regularizers
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.optimizers import Adam

# Configurações 
print(tf.config.list_physical_devices())
BATCH_SIZE = 64

KeyboardInterrupt: 

# 1.0 - Classificação de Presença Baseada em Fatores Socioeconômicos Usando Redes Neurais

In [ ]:
colunas = ['Q001','Q002','Q003', 'Q004', 'Q005', 'Q006', 'Q007', 'Q008', 'Q009', 'Q010', 'Q011', 'Q012', 'Q013', 'Q014', 'Q015', 'Q016', 'Q017', 'Q018', 
           'Q019', 'Q020', 'Q021', 'Q022', 'Q023', 'Q024', 'Q025', 'TP_PRESENCA_LC', 'TP_PRESENCA_CH', 'TP_PRESENCA_CN', 'TP_PRESENCA_MT', 
           'TP_FAIXA_ETARIA', 'TP_SEXO','TP_ESTADO_CIVIL', 'TP_COR_RACA', 'TP_ESCOLA', 'TP_ST_CONCLUSAO', 'IN_TREINEIRO', 
           'NU_ANO', 'TP_LOCALIZACAO_ESC','TP_SIT_FUNC_ESC', 'NU_NOTA_CN', 'NU_NOTA_CH', 'NU_NOTA_LC', 'NU_NOTA_MT', 'NU_NOTA_REDACAO', 'TP_DEPENDENCIA_ADM_ESC']

df = pd.read_parquet("../data/enem_parquet", columns = colunas)

## 1.2- Pré-Processando os Dados

In [ ]:
df = df.sample(n=50_000, random_state=42)

## 1.2- Limpando os Dados

In [8]:
FALTOU = (
    (df['TP_PRESENCA_CH'] != 1) | 
    (df['TP_PRESENCA_LC'] != 1) | 
    (df['TP_PRESENCA_CN'] != 1) | 
    (df['TP_PRESENCA_MT'] != 1)
)

df['FALTOU'] = FALTOU.astype(int)

In [9]:
df = df[df['FALTOU'] == 0]

In [10]:
df = df[df['TP_ESCOLA'].isin([2,3])]
df = df[df['TP_ESTADO_CIVIL'].isin([1,2,3,4])]

In [11]:
df['TP_LOCALIZACAO_ESC'] = df['TP_LOCALIZACAO_ESC'].fillna(0)
df['TP_DEPENDENCIA_ADM_ESC'] = df['TP_DEPENDENCIA_ADM_ESC'].fillna(0)
df['TP_SIT_FUNC_ESC'] = df['TP_SIT_FUNC_ESC'].fillna(0)

In [12]:
questionario = [f'Q{str(i).zfill(3)}' for i in range(1, 26) if i != 5]
df = df.dropna(subset=questionario)

## 1.3 - Criando Rótulo

In [13]:
df['MEDIA'] = (df['NU_NOTA_CN'] + df['NU_NOTA_CH'] + df['NU_NOTA_MT']+  df['NU_NOTA_LC'] + df['NU_NOTA_REDACAO']) / 5

df['CLASSE'] = df.groupby('NU_ANO')['MEDIA'].transform(
    lambda x: pd.qcut(x, q=2, labels=[0,1])
).astype(int)
    
df['CLASSE'] = df['CLASSE'].astype(int)

## 1.4 - Tratando os Dados

In [14]:
nominais = ['TP_ESCOLA', 'TP_DEPENDENCIA_ADM_ESC',
            'TP_ESTADO_CIVIL', 'TP_LOCALIZACAO_ESC', 'TP_SIT_FUNC_ESC']

ordinais = ['TP_FAIXA_ETARIA', 'TP_ST_CONCLUSAO']

categorias_quest = []
for i in range(1, 26):
    if i == 5:
        continue
    elif i == 6:
        categorias_quest.append(list('ABCDEFGHIJKLMNOPQ'))  
    elif i == 25:
        categorias_quest.append(list('AB'))                  
    elif i in [1, 2]:
        categorias_quest.append(list('ABCDEFGH'))            
    elif i in [3, 4]:
        categorias_quest.append(list('ABCDEF'))              
    else:
        categorias_quest.append(list('ABCDE'))               

binarias = ['IN_TREINEIRO']

In [15]:
pipe_nominal = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

pipe_ordinal = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)

pipe_quest = OrdinalEncoder(
    categories=categorias_quest,
    handle_unknown='use_encoded_value',
    unknown_value=-1
)

pipe_continua = MinMaxScaler()

preprocessor = ColumnTransformer(transformers=[
    ('nominal',      pipe_nominal,  nominais),
    ('ordinal',      pipe_ordinal,  ordinais),
    ('questionario', pipe_quest,    questionario),
    ('binaria',      'passthrough', binarias),
], remainder='drop')


## 1.3- Construção da Matriz X e Vetor y

In [16]:
X = df.drop(columns=['MEDIA', 'CLASSE', 'FALTOU'])
y = df['CLASSE']

## 1.4 - Separação em Dados de Treino, Validação e Teste

In [17]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

In [18]:
## 1.5 - Tratando os Dados

In [31]:
max_neurons = num_max_neuronio(X_train, d = X_train.shape[1])
print("Número máximo de neurônios:", max_neurons)

Número máximo de neurônios: 14


In [ ]:
def create_model(input_dim, neurons, learning_rate, l2_reg, dropout):
    model = Sequential()
    model.input_shape = (input_dim,)
    
    model.add(Dense(neurons,
                    input_dim=input_dim,
                    kernel_initializer='he_normal',
                    kernel_regularizer=regularizers.l2(l2_reg),
                    activation='relu'))
    if dropout > 0:
        model.add(Dropout(dropout))
    model.add(Dense(1, kernel_initializer='he_normal', activation='sigmoid'))
    
    model.compile(
        loss='binary_crossentropy',
        optimizer=Adam(learning_rate=learning_rate),
        metrics=['accuracy']
    )
    return model

In [33]:
param_grid = {
    'neurons':       [13],
    'learning_rate': [0.1, 0.01, 0.001, 0.0001],   
    'batch_size':    [32,64],             
    'epochs':        [100],
    'l2_reg':        [0.001, 0.01, 0.1],          
    'dropout':       [0.0, 0.2],
}

# Gera todas as combinações
keys, values = zip(*param_grid.items())
combinacoes = [dict(zip(keys, v)) for v in itertools.product(*values)]
print(f"Total de combinações: {len(combinacoes)}")

skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

resultados = []

for params in combinacoes:
    print(f"Testando: {params}")
    
    accs = []

    for train_idx, val_idx in skf.split(X_train, y_train):
        X_tr, X_val_fold = X_train[train_idx], X_train[val_idx]
        y_tr, y_val_fold = y_train.iloc[train_idx], y_train.iloc[val_idx]

        model = create_model(
            input_dim=X_train.shape[1],
            neurons=params['neurons'],
            learning_rate=params['learning_rate'],
            l2_reg=params['l2_reg'],
            dropout=params['dropout']
        )

        early_stop = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)

        history = model.fit(
            X_tr, y_tr,
            validation_data=(X_val_fold, y_val_fold),
            epochs=params['epochs'],  
            batch_size=params['batch_size'],
            callbacks=[early_stop],
            verbose=1
        )

        # Avaliação no fold
        loss, acc = model.evaluate(X_val_fold, y_val_fold, verbose=0)
        accs.append(acc)

        # Libera memória (importante no Keras)
        del model
        gc.collect()

    mean_acc = np.mean(accs)

    resultados.append({
        'params': params,
        'mean_accuracy': mean_acc
    })

    print(f"Accuracy média: {mean_acc:.4f}")

Total de combinações: 48
Testando: {'neurons': 13, 'learning_rate': 0.1, 'batch_size': 32, 'epochs': 100, 'l2_reg': 0.001, 'dropout': 0.0}
Epoch 1/100
 11/146 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.5599 - loss: 1.2291

c:\Users\sergi\OneDrive\Documentos\GitHub\Projeto_ENEM\venv\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


146/146 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - accuracy: 0.6043 - loss: 0.7201 - val_accuracy: 0.7080 - val_loss: 0.6247
Epoch 2/100
146/146 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.6844 - loss: 0.6215 - val_accuracy: 0.5372 - val_loss: 0.6974
Epoch 3/100
146/146 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.6805 - loss: 0.6235 - val_accuracy: 0.6991 - val_loss: 0.5927
Epoch 4/100
146/146 ━━━━━━━━━━━━━━━━━━━━ 3s 18ms/step - accuracy: 0.6758 - loss: 0.6192 - val_accuracy: 0.6725 - val_loss: 0.6217
Epoch 5/100
146/146 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.6848 - loss: 0.6076 - val_accuracy: 0.7072 - val_loss: 0.5851
Epoch 6/100
146/146 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.6741 - loss: 0.6162 - val_accuracy: 0.6995 - val_loss: 0.6085
Epoch 7/100
146/146 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.6878 - loss: 0.6071 - val_accuracy: 0.7021 - val_loss: 0.5902
Epoch 8/100
146/146 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.6816 - loss: 0.6075 - val_accuracy

c:\Users\sergi\OneDrive\Documentos\GitHub\Projeto_ENEM\venv\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


146/146 ━━━━━━━━━━━━━━━━━━━━ 3s 18ms/step - accuracy: 0.6310 - loss: 0.6992 - val_accuracy: 0.7106 - val_loss: 0.5985
Epoch 2/100
146/146 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step - accuracy: 0.6415 - loss: 0.6456 - val_accuracy: 0.6670 - val_loss: 0.6219
Epoch 3/100
146/146 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.6516 - loss: 0.6436 - val_accuracy: 0.7046 - val_loss: 0.6018
Epoch 4/100
146/146 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.6334 - loss: 0.6466 - val_accuracy: 0.7149 - val_loss: 0.5919
Epoch 5/100
146/146 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step - accuracy: 0.6167 - loss: 0.6562 - val_accuracy: 0.6665 - val_loss: 0.6225
Epoch 6/100
146/146 ━━━━━━━━━━━━━━━━━━━━ 3s 18ms/step - accuracy: 0.6392 - loss: 0.6477 - val_accuracy: 0.7068 - val_loss: 0.5991
Epoch 7/100
146/146 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.6456 - loss: 0.6433 - val_accuracy: 0.7106 - val_loss: 0.5948
Epoch 8/100
146/146 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.5600 - loss: 0.6806 - val_accuracy

...

73/73 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.5233 - loss: 1.0350 - val_accuracy: 0.5625 - val_loss: 0.9432
Epoch 2/100
73/73 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.5426 - loss: 0.9669 - val_accuracy: 0.5912 - val_loss: 0.8951
Epoch 3/100
73/73 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.5600 - loss: 0.9265 - val_accuracy: 0.6190 - val_loss: 0.8687
Epoch 4/100
73/73 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.5767 - loss: 0.9020 - val_accuracy: 0.6271 - val_loss: 0.8505
Epoch 5/100
73/73 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.5904 - loss: 0.8861 - val_accuracy: 0.6404 - val_loss: 0.8346
Epoch 6/100
73/73 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.5878 - loss: 0.8738 - val_accuracy: 0.6503 - val_loss: 0.8203
Epoch 7/100
73/73 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.6133 - loss: 0.8489 - val_accuracy: 0.6567 - val_loss: 0.8069
Epoch 8/100
73/73 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.6116 - loss: 0.8387 - val_accuracy: 0.6622 - val_l

c:\Users\sergi\OneDrive\Documentos\GitHub\Projeto_ENEM\venv\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


73/73 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.5111 - loss: 3.6339 - val_accuracy: 0.5081 - val_loss: 3.4875
Epoch 2/100
73/73 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.5266 - loss: 3.3785 - val_accuracy: 0.5244 - val_loss: 3.2746
Epoch 3/100
73/73 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.5383 - loss: 3.1938 - val_accuracy: 0.5458 - val_loss: 3.1125
Epoch 4/100
73/73 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.5441 - loss: 3.0445 - val_accuracy: 0.5578 - val_loss: 2.9737
Epoch 5/100
73/73 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.5619 - loss: 2.9122 - val_accuracy: 0.5616 - val_loss: 2.8470
Epoch 6/100
73/73 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.5717 - loss: 2.7906 - val_accuracy: 0.5766 - val_loss: 2.7295
Epoch 7/100
73/73 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.5752 - loss: 2.6772 - val_accuracy: 0.5916 - val_loss: 2.6190
Epoch 8/100
73/73 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.5850 - loss: 2.5703 - val_accuracy: 0.5989 - val_l

c:\Users\sergi\OneDrive\Documentos\GitHub\Projeto_ENEM\venv\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


73/73 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.5049 - loss: 4.2419 - val_accuracy: 0.4970 - val_loss: 3.9807
Epoch 2/100
73/73 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.4985 - loss: 3.8144 - val_accuracy: 0.4932 - val_loss: 3.5847
Epoch 3/100
73/73 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.4979 - loss: 3.4952 - val_accuracy: 0.5090 - val_loss: 3.2855
Epoch 4/100
73/73 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.5158 - loss: 3.2566 - val_accuracy: 0.5197 - val_loss: 3.0696
Epoch 5/100
73/73 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.5270 - loss: 3.0735 - val_accuracy: 0.5462 - val_loss: 2.9050
Epoch 6/100
73/73 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.5343 - loss: 2.9280 - val_accuracy: 0.5801 - val_loss: 2.7715
Epoch 7/100
73/73 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.5525 - loss: 2.8018 - val_accuracy: 0.6074 - val_loss: 2.6562
Epoch 8/100
73/73 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.5675 - loss: 2.6836 - val_accuracy: 0.6250 - val_l

In [34]:
resultados_df = pd.DataFrame(resultados)
resultados_df = resultados_df.sort_values(by='mean_accuracy', ascending=False)

In [ ]:
resultados_df[:1] 

,params,mean_accuracy
27,"{'neurons': 13, 'learning_rate': 0.001, 'batch...",0.713244


In [38]:
resultados_df['params'].iloc[0]

{'neurons': 13,
 'learning_rate': 0.001,
 'batch_size': 32,
 'epochs': 100,
 'l2_reg': 0.01,
 'dropout': 0.2}

In [41]:
resultados_df.to_csv("resultados_fatores_socioeconomicos.csv", index=False)

In [15]:
X = df[colunas]
y = df['CLASSE']

In [16]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [17]:
preprocessor.fit(X_train)

X_train = preprocessor.transform(X_train).astype(np.float32)

In [18]:
model = Sequential()

model.add(Dense(4837, input_dim=X_train.shape[1], kernel_initializer='he_normal', activation='relu', kernel_regularizer=regularizers.l2(0.001)))
model.add(Dropout(0.2))

model.add(Dense(1, kernel_initializer='he_normal', activation='sigmoid'))

model.compile(loss='binary_crossentropy', optimizer=Adam(learning_rate=0.001), metrics=['accuracy'])

history = model.fit(
    X_train, y_train,
    epochs=100,
    batch_size= BATCH_SIZE,
    verbose=1
)

C:\Users\Micael\AppData\Roaming\Python\Python312\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/100
2732/2732 ━━━━━━━━━━━━━━━━━━━━ 17s 5ms/step - accuracy: 0.6911 - loss: 1.5490
Epoch 2/100
2732/2732 ━━━━━━━━━━━━━━━━━━━━ 14s 5ms/step - accuracy: 0.7052 - loss: 0.5853
Epoch 3/100
2732/2732 ━━━━━━━━━━━━━━━━━━━━ 13s 5ms/step - accuracy: 0.7062 - loss: 0.5708
Epoch 4/100
2732/2732 ━━━━━━━━━━━━━━━━━━━━ 13s 5ms/step - accuracy: 0.7078 - loss: 0.5678
Epoch 5/100
2732/2732 ━━━━━━━━━━━━━━━━━━━━ 13s 5ms/step - accuracy: 0.7073 - loss: 0.5674
Epoch 6/100
2732/2732 ━━━━━━━━━━━━━━━━━━━━ 19s 7ms/step - accuracy: 0.7065 - loss: 0.5671
Epoch 7/100
2732/2732 ━━━━━━━━━━━━━━━━━━━━ 16s 6ms/step - accuracy: 0.7074 - loss: 0.5662
Epoch 8/100
2732/2732 ━━━━━━━━━━━━━━━━━━━━ 19s 5ms/step - accuracy: 0.7078 - loss: 0.5662
Epoch 9/100
2732/2732 ━━━━━━━━━━━━━━━━━━━━ 13s 5ms/step - accuracy: 0.7082 - loss: 0.5661
Epoch 10/100
2732/2732 ━━━━━━━━━━━━━━━━━━━━ 14s 5ms/step - accuracy: 0.7083 - loss: 0.5655
Epoch 11/100
2732/2732 ━━━━━━━━━━━━━━━━━━━━ 14s 5ms/step - accuracy: 0.7084 - loss: 0.5659
Epoch 12

In [19]:
X_test  = preprocessor.transform(X_test).astype(np.float32)

In [20]:
y_pred = (model.predict(X_test) > 0.5).astype(int).flatten()
print(classification_report(y_test, y_pred))

1366/1366 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step
              precision    recall  f1-score   support

           0       0.69      0.78      0.73     21901
           1       0.74      0.64      0.69     21798

    accuracy                           0.71     43699
   macro avg       0.72      0.71      0.71     43699
weighted avg       0.72      0.71      0.71     43699



## 1.7 - Treinando a Rede Neural

In [ ]:
model = Sequential()

model.add(Dense(256, input_dim=x_train.shape[1], kernel_initializer='he_normal', activation='relu'))
model.add(BatchNormalization())
model.add(Dropout(0.2))

model.add(Dense(128, input_dim=x_train.shape[1], kernel_initializer='he_normal', activation='relu'))
model.add(BatchNormalization())
model.add(Dropout(0.2))

model.add(Dense(64, kernel_initializer='he_normal', activation='relu'))
model.add(BatchNormalization())
model.add(Dropout(0.2))

model.add(Dense(1, kernel_initializer='he_normal', activation='sigmoid'))

model.compile(loss='binary_crossentropy', optimizer=Adam(), metrics=['accuracy'])


es = EarlyStopping(monitor='val_loss', mode='min', patience=10, verbose=1)

history = model.fit(
    x_train, y_train,
    epochs=100,
    batch_size= BATCH_SIZE,
    validation_data= (x_val, y_val),
    callbacks=[es],
    verbose=1
)

Epoch 1/100
137/137 ━━━━━━━━━━━━━━━━━━━━ 5s 16ms/step - accuracy: 0.6788 - loss: 0.6116 - val_accuracy: 0.7062 - val_loss: 0.5668
Epoch 2/100
137/137 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - accuracy: 0.7010 - loss: 0.5737 - val_accuracy: 0.7119 - val_loss: 0.5583
Epoch 3/100
137/137 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.7070 - loss: 0.5661 - val_accuracy: 0.7112 - val_loss: 0.5570
Epoch 4/100
137/137 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - accuracy: 0.7086 - loss: 0.5630 - val_accuracy: 0.7122 - val_loss: 0.5557
Epoch 5/100
137/137 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - accuracy: 0.7100 - loss: 0.5606 - val_accuracy: 0.7136 - val_loss: 0.5560
Epoch 6/100
137/137 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - accuracy: 0.7112 - loss: 0.5589 - val_accuracy: 0.7128 - val_loss: 0.5548
Epoch 7/100
137/137 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - accuracy: 0.7123 - loss: 0.5581 - val_accuracy: 0.7130 - val_loss: 0.5551
Epoch 8/100
137/137 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - accuracy: 0.7129 - loss: 0.5564 - 

## 1.8 - Exibindo Resultados no Teste

In [ ]:
y_pred = (model.predict(x_test) > 0.5).astype(int).flatten()
print(classification_report(y_test, y_pred))

1366/1366 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step
              precision    recall  f1-score   support

           0       0.70      0.76      0.73     21901
           1       0.74      0.66      0.70     21798

    accuracy                           0.71     43699
   macro avg       0.72      0.71      0.71     43699
weighted avg       0.72      0.71      0.71     43699



## Seção de Testes

In [ ]:
import numpy as np
import pandas as pd

#np.random.seed(42)  # para reprodutibilidade

x_aluno = pd.DataFrame([{
    # Questionário socioeconômico
    'Q001': np.random.choice(list('ABCDEFGH')),
    'Q002': np.random.choice(list('ABCDEFGH')),
    'Q003': np.random.choice(list('ABCDEF')),
    'Q004': np.random.choice(list('ABCDEF')),
    'Q006': np.random.choice(list('ABCDEFGHIJKLMNOPQ')),
    'Q007': np.random.choice(list('ABCD')),
    'Q008': np.random.choice(list('ABCDE')),
    'Q009': np.random.choice(list('ABCDE')),
    'Q010': np.random.choice(list('ABCDE')),
    'Q011': np.random.choice(list('ABCDE')),
    'Q012': np.random.choice(list('ABCDE')),
    'Q013': np.random.choice(list('ABCDE')),
    'Q014': np.random.choice(list('ABCDE')),
    'Q015': np.random.choice(list('ABCDE')),
    'Q016': np.random.choice(list('ABCDE')),
    'Q017': np.random.choice(list('ABCDE')),
    'Q018': np.random.choice(list('ABCDE')),
    'Q019': np.random.choice(list('ABCDE')),
    'Q020': np.random.choice(list('ABCDE')),
    'Q021': np.random.choice(list('ABCDE')),
    'Q022': np.random.choice(list('ABCDE')),
    'Q023': np.random.choice(list('ABCDE')),
    'Q024': np.random.choice(list('ABCDE')),
    'Q025': np.random.choice(list('AB')),

    # Categóricas
    'TP_FAIXA_ETARIA':        np.random.choice([3,4,2,1,14,6,5,12,8,7,11,9,13,16,10,17,15,19,18,20]),
    'TP_ST_CONCLUSAO':        2,
    'TP_ESCOLA':              np.random.choice([2, 3]),
    'TP_DEPENDENCIA_ADM_ESC': np.random.choice([0., 4., 2., 1., 3.]),
    'TP_ESTADO_CIVIL':        np.random.choice([1, 2, 3, 4]),
    'TP_LOCALIZACAO_ESC':     np.random.choice([0., 1., 2.]),
    'TP_SIT_FUNC_ESC':        np.random.choice([0., 1., 3., 4., 2.]),

    # Binária
    'IN_TREINEIRO': np.random.choice([0, 1]),
}])

# Transformar com o preprocessor já treinado
x_aluno_transformado = preprocessor.transform(x_aluno).astype(np.float32)

# Probabilidade
prob = model.predict(x_aluno_transformado).flatten()[0]
print(f"Probabilidade de desempenho alto: {prob:.1%}")

# Nota estimada
media_classe_0 = df[df['CLASSE'] == 0]['MEDIA'].mean()
media_classe_1 = df[df['CLASSE'] == 1]['MEDIA'].mean()
nota_estimada = (1 - prob) * media_classe_0 + prob * media_classe_1
print(f"Nota estimada: {nota_estimada:.1f}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
Probabilidade de desempenho alto: 8.7%
Nota estimada: 470.7
